# Install necessary tools for this script

In [2]:
!git clone https://github.com/opengrep/opengrep-rules.git
!curl -fsSL https://raw.githubusercontent.com/opengrep/opengrep/main/install.sh | bash
!pip install kagglehub[pandas-datasets]

fatal: destination path 'opengrep-rules' already exists and is not an empty directory.


Go to https:/github.com/sigstore/cosign to install it.
Destination binary /home/regularpooria/.opengrep/cli/v1.14.1/opengrep already exists.
Updated symlink from /home/regularpooria/.opengrep/cli/latest/opengrep to point to /home/regularpooria/.opengrep/cli/v1.14.1/opengrep.

To launch Opengrep now, type:
opengrep


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
import json
import shutil
import csv
import pandas
import re


In [2]:

df = pandas.read_parquet("hf://datasets/regularpooria/wildcode_conversation_redo/data/train-00000-of-00001.parquet")
df

/home/regularpooria/Projects/WildCode/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,conversation_hash,language,user_text,rule_id,vuln_message,file_uri,start_line,sarif_file,prompt_variant,model,...,output_version,response_text,status,error,input_tokens_est,input_tokens,output_tokens,total_tokens,variant,security_instruction
0,f70f171081d6ecf4b3863b0fbb7577c3,English,Create a step-by-step guide for building a Bus...,files.javascript.rules.express.security.audit....,A CSRF middleware was not detected in your exp...,files/javascript/codes/f70f171081d6ecf4b3863b0...,6,output.sarif,as_security_specialist,gpt-4o-mini,...,output_1200,Building a Business Directory Listing app invo...,OK,NaN,158,117,1158,1275,act_as_security_specialist,
1,86ea16227c57d015ff8dfd37d9dbd370,English,My app is business directory listing based on ...,files.javascript.rules.express.security.audit....,A CSRF middleware was not detected in your exp...,files/javascript/codes/86ea16227c57d015ff8dfd3...,6,output.sarif,as_security_specialist,gpt-4o-mini,...,output_1200,Sure! Let's dive into the database design for ...,OK,NaN,134,107,1168,1275,act_as_security_specialist,
2,e707046ddb90bb77ad490983a7331415,English,write mongoDB schema on node.js for chat\nwrit...,files.javascript.rules.express.security.audit....,A CSRF middleware was not detected in your exp...,files/javascript/codes/e707046ddb90bb77ad49098...,5,output.sarif,as_security_specialist,gpt-4o-mini,...,output_1200,Creating a MongoDB schema for a chat applicati...,OK,NaN,57,57,769,826,act_as_security_specialist,
3,dd42a864f91ddaaeafbc45667656f3b8,English,import json\nimport requests\n\n# Server addre...,files.python.rules.lang.security.audit.insecur...,Detected a request using 'http://'. This reque...,files/python/codes/dd42a864f91ddaaeafbc4566765...,32,output.sarif,as_security_specialist,gpt-4o-mini,...,output_1200,The error message you're encountering indicate...,OK,NaN,297,367,715,1082,act_as_security_specialist,
4,dd42a864f91ddaaeafbc45667656f3b8,English,import json\nimport requests\n\n# Server addre...,files.python.rules.lang.security.audit.insecur...,Detected a request using 'http://'. This reque...,files/python/codes/dd42a864f91ddaaeafbc4566765...,15,output.sarif,as_security_specialist,gpt-4o-mini,...,output_1200,The error message you're encountering indicate...,OK,NaN,297,367,694,1061,act_as_security_specialist,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23057,e5400d3292dd24d094a013d86f088338,English,انا لدي كود البايثون ادناه ، اريد اضافة للكود ...,files.python.rules.lang.security.audit.subproc...,Found 'subprocess' function 'run' with 'shell=...,files/python/codes/e5400d3292dd24d094a013d86f0...,37,output.sarif,vuln_specific_prof,gpt-4o-mini,...,output_800,بالطبع! سأقوم بتعديل الكود الخاص بك لإضافة تثب...,OK,NaN,956,947,800,1747,varies_according_to_vulnerability,Ensure that the code follows secure coding bes...
23058,e5400d3292dd24d094a013d86f088338,English,انا لدي كود البايثون ادناه ، اريد اضافة للكود ...,files.python.rules.lang.security.audit.dangero...,Detected subprocess function 'run' without a s...,files/python/codes/e5400d3292dd24d094a013d86f0...,58,output.sarif,vuln_specific_prof,gpt-4o-mini,...,output_800,بالطبع! سأقوم بتعديل الكود الخاص بك لإضافة تثب...,OK,NaN,956,947,800,1747,varies_according_to_vulnerability,Ensure that the code follows secure coding bes...
23059,e5400d3292dd24d094a013d86f088338,English,انا لدي كود البايثون ادناه ، اريد اضافة للكود ...,files.python.rules.lang.security.audit.subproc...,Found 'subprocess' function 'run' with 'shell=...,files/python/codes/e5400d3292dd24d094a013d86f0...,58,output.sarif,vuln_specific_prof,gpt-4o-mini,...,output_800,بالطبع! سأقوم بتعديل الكود الخاص بك لإضافة تثب...,OK,NaN,956,947,800,1747,varies_according_to_vulnerability,Ensure that the code follows secure coding bes...
23060,e5400d3292dd24d094a013d86f088338,English,انا لدي كود البايثون ادناه ، اريد اضافة للكود ...,files.python.rules.lang.security.audit.dangero...,Detected subprocess function 'run' without a s...,files/python/co

# Turning dataset into actual files

In [3]:
!rm -rf ../../files
!mkdir ../../files

In [4]:
def extract_code_snippets(text):
    """
    Extracts all code blocks from Markdown text.
    Returns a list of tuples: (language, code)
    Language is None if not specified.
    Handles consecutive code blocks correctly.
    """
    blocks = []
    
    # Updated regex pattern to handle various markdown formats:
    # ``` followed by optional language, then newline
    # Content until closing ``` at start of line or end of string
    pattern = r"```(\w+)?\n?(.*?)\n?```"
    
    matches = re.findall(pattern, text, re.DOTALL)
    
    for lang, code in matches:
        # Clean up the code content
        cleaned_code = code.strip()
        if cleaned_code:  # Only add non-empty code blocks
            blocks.append((lang if lang else None, cleaned_code))
    
    return blocks

In [5]:
languages = [
    "c",
    "csharp",
    "java",
    "javascript",
    "php",
    "python",
]

banned_langs = [
    "bash"
]

MODES = [
    "varies_according_to_vulnerability",
    "from_security_standpoint",
    "act_as_security_specialist",
    "original"
]

for language in languages:
    os.makedirs(f"../../files/{language}/codes/", exist_ok=True)
    os.makedirs(f"../../files/{language}/rules/", exist_ok=True)
    for mode in MODES:
        os.makedirs(f"../../files/{language}/codes/{mode}", exist_ok=True)
        

tracker = {
    "c": 0,
    "csharp": 0,
    "java": 0,
    "javascript": 0,
    "php": 0,
    "python": 0,
}
    
for i, row in df[df["max_output_tokens"] == 1200].iterrows():
    code_blocks = extract_code_snippets(row["response_text"])
    conversation_hash = row["conversation_hash"]
    extension = row["file_uri"].split(".")[-1]
    max_output_tokens = row["max_output_tokens"]
    variant = row["variant"]
    for index, block in enumerate(code_blocks):
        filename = "../../"+"/".join(row["file_uri"].split("/")[:-1]) + f"/{variant}/{conversation_hash}_{index}_{max_output_tokens}_{variant.replace("_", "-")}.{extension}"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(block[1])
    
    

In [6]:
def copy_security_yaml_rules(src_root: str, dst_root: str):
    """
    Walk src_root, find all .yaml files under any 'security' folder,
    and copy them to dst_root, preserving subdirectory structure.
    """
    for root, dirs, files in os.walk(src_root):
        # only consider paths that have 'security' in their hierarchy
        if 'security' in root.split(os.sep):
            for file in files:
                if file.endswith('.yaml'):
                    # compute relative path under src_root
                    rel_dir = os.path.relpath(root, src_root)
                    dst_dir = os.path.join(dst_root, rel_dir)
                    os.makedirs(dst_dir, exist_ok=True)

                    src_file = os.path.join(root, file)
                    dst_file = os.path.join(dst_dir, file)
                    shutil.copy2(src_file, dst_file)
                    print(f"Copied: {rel_dir}/{file}")

In [7]:
for language in languages:
    if not os.path.exists(f"../../opengrep-rules/{language}"):
        print(f"Could not find path: ../../opengrep-rules/{language}")
        break
    copy_security_yaml_rules(f"../../opengrep-rules/{language}", f"../../files/{language}/rules/")

Copied: lang/security/double-free.yaml
Copied: lang/security/function-use-after-free.yaml
Copied: lang/security/info-leak-on-non-formatted-string.yaml
Copied: lang/security/insecure-use-gets-fn.yaml
Copied: lang/security/insecure-use-memset.yaml
Copied: lang/security/insecure-use-printf-fn.yaml
Copied: lang/security/insecure-use-scanf-fn.yaml
Copied: lang/security/insecure-use-strcat-fn.yaml
Copied: lang/security/insecure-use-string-copy-fn.yaml
Copied: lang/security/insecure-use-strtok-fn.yaml
Copied: lang/security/random-fd-exhaustion.yaml
Copied: lang/security/use-after-free.yaml
Copied: dotnet/security/mvc-missing-antiforgery.yaml
Copied: dotnet/security/net-webconfig-debug.yaml
Copied: dotnet/security/net-webconfig-trace-enabled.yaml
Copied: dotnet/security/razor-template-injection.yaml
Copied: dotnet/security/use_deprecated_cipher_algorithm.yaml
Copied: dotnet/security/use_ecb_mode.yaml
Copied: dotnet/security/use_weak_rng_for_keygeneration.yaml
Copied: dotnet/security/use_weak_r

In [ ]:
for language in languages:
    for mode in MODES:
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/{mode}_1200.sarif -f ../../files/{language}/rules ../../files/{language}/codes/{mode}


┌──────────────┐
│ Opengrep CLI │
└──────────────┘

                                                                                
Scanning 372 files with 12 Code rules:
            
  CODE RULES
  Scanning 372 files with 12 c rules.
          
  PROGRESS
   
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00                                                                                ━━━━━━━━━━━━━━━━━━━━━━━━   0% -:--:--
                     
                     
┌───────────────────┐
│ 225 Code Findings │
└───────────────────┘
                                                                                
                                                                    
  ../../files/c/codes/varies_according_to_vulnerability/0156943a01daf38901b7b87f
  9e07e13_2_1200_varies-according-to-vulnerability.c                         
    ❯❱ files.c.rules.lang.security.insecure-use-string-copy-fn
          Finding triggers whenever there is a strcpy or strncpy used.
          T

In [8]:
for language in languages:
    for mode in MODES:
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/{mode}_hash_conversation_redo_.sarif -f ../../opengrep-rules/all_langs/hash ../../files/{language}/codes/{mode}
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/{mode}_sql_conversation_redo_.sarif -f ../../opengrep-rules/all_langs/sql ../../files/{language}/codes/{mode}
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/{mode}_random_conversation_redo_.sarif -f ../../opengrep-rules/all_langs/random ../../files/{language}/codes/{mode}
        !opengrep scan --no-git-ignore --sarif-output=../../files/{language}/{mode}_deserialization_conversation_redo_.sarif -f ../../opengrep-rules/all_langs/deserialization ../../files/{language}/codes/{mode}



┌──────────────┐
│ Opengrep CLI │
└──────────────┘

                                                                                
Scanning 372 files with 1 Code rule:
            
  CODE RULES
  Scanning 372 files.
          
  PROGRESS
   
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00                                                                                ━━━━━━━━━━━━━━━━━━━━━━━━   0% -:--:--
                
                
┌──────────────┐
│ Scan Summary │
└──────────────┘

Ran 1 rule on 372 files: 0 findings.

┌──────────────┐
│ Opengrep CLI │
└──────────────┘

                                                                                
Scanning 372 files with 2 Code rules:
            
  CODE RULES
  Scanning 372 files with 2 c rules.
          
  PROGRESS
   
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00                                                                                ━━━━━━━━━━━━━━━━━━━━━━━━   0% -:--:--
                  
      

# IGNORE CELLS BELOW

# Running linting on the files to remove mislabeled files

In [8]:
import os
import json
import subprocess
from tqdm import tqdm
import sys
import re

In [ ]:
env = os.environ.copy()
env["PATH"] = "/home/regularpooria/.nvm/versions/node/v22.17.1/bin:" + env["PATH"]
env["PATH"] = os.path.expanduser("~/.dotnet/tools") + ":" + env["PATH"]

subprocess.run(["dotnet-script", "--version"], env=env)

FileNotFoundError: [Errno 2] No such file or directory: 'dotnet script'

In [10]:
import py_compile

directory = "../../files/python/codes/"
output_file = "../../results/python_lint_conversation_redo.json"
all_results = []

py_files = [f for f in os.listdir(directory) if f.endswith(".py")]
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Disable .pyc creation globally
sys.dont_write_bytecode = True

for filename in tqdm(py_files, desc="Linting files", unit="file"):
    filepath = os.path.join(directory, filename)

    try:
        # Compile with syntax check, no .pyc files written
        py_compile.compile(filepath, doraise=True)
    except py_compile.PyCompileError as e:
        err = {
            "type": "syntax-error",
            "module": filename,
            "obj": "",
            "line": getattr(e.exc_value, "lineno", None),
            "column": None,
            "path": filepath,
            "symbol": "syntax-error",
            "message": str(e.exc_value),
            "message-id": ""
        }
        all_results.append(err)

# Write all results to JSON once after all files are processed
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)


Linting files:  10%|█         | 1182/11543 [00:00<00:01, 6120.46file/s]../../files/python/codes/305b5f10cd8843fa7c2e8b3476e243ca_3_1200_from-security-standpoint.py:1: SyntaxWarning: invalid escape sequence '\p'
  "C:\path\to\your\script\reset_audio_clips.bat" "%V"
Linting files:  16%|█▌        | 1866/11543 [00:00<00:01, 6439.44file/s]../../files/python/codes/76481adc0116753c8d5fa13b8c06519f_2_1200_from-security-standpoint.py:1: SyntaxWarning: invalid escape sequence '\T'
  Start-Process -FilePath "path\to\ThinAppInstaller.exe" -Wait
../../files/python/codes/e39f94a452031461327f6bef2f915f70_0_1200_from-security-standpoint.py:8: SyntaxWarning: invalid escape sequence '\F'
  <path>D:\Files\Captcha_Training\Dataset\1m6=x.png</path>
Linting files:  29%|██▉       | 3322/11543 [00:00<00:01, 6970.93file/s]../../files/python/codes/305b5f10cd8843fa7c2e8b3476e243ca_2_1200_original.py:1: SyntaxWarning: invalid escape sequence '\P'
  "C:\Path\To\Python\python.exe" "C:\Path\To\Your\Script\reset_audi

In [ ]:
import os
import json
import subprocess

directory = "files/javascript/codes/"
output_file = "../../results/js_lint_conversation_redo.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

js_files = [os.path.join(directory, f) for f in os.listdir("../../"+directory) if f.endswith(".js")]
all_results = []


# Batch size (tweak if needed)
BATCH_SIZE = 1000

def run_eslint_batch(batch_files):
    result = subprocess.run(
        [
            "eslint",
            "--config",
            "linting_rules/js.eslint.config.mjs",
            "--format",
            "json",
            *batch_files,
        ],
        capture_output=True,
        text=True,
        env=env,
        cwd="../../"
    )
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        print("⚠️ Failed to parse JSON from ESLint output:")
        print(result.stderr)
        return []

# Run in batches
for i in range(0, len(js_files), BATCH_SIZE):
    batch = js_files[i:i + BATCH_SIZE]
    file_results = run_eslint_batch(batch)

    for file_result in file_results:
        filename = os.path.basename(file_result.get("filePath", ""))
        for msg in file_result.get("messages", []):
            if msg.get("fatal", False) or msg.get("severity", 0) == 2:
                all_results.append({
                    "type": "syntax-error",
                    "module": filename,
                    "obj": "",
                    "line": msg.get("line"),
                    "column": msg.get("column"),
                    "path": file_result.get("filePath"),
                    "symbol": "syntax-error",
                    "message": msg.get("message", ""),
                    "message-id": "",
                })

# Save results
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"✅ Linting complete, results saved to {output_file}")


['files/javascript/codes/f70f171081d6ecf4b3863b0fbb7577c3_0_1200_act-as-security-specialist.js', 'files/javascript/codes/f70f171081d6ecf4b3863b0fbb7577c3_1_1200_act-as-security-specialist.js', 'files/javascript/codes/f70f171081d6ecf4b3863b0fbb7577c3_2_1200_act-as-security-specialist.js', 'files/javascript/codes/f70f171081d6ecf4b3863b0fbb7577c3_3_1200_act-as-security-specialist.js', 'files/javascript/codes/86ea16227c57d015ff8dfd37d9dbd370_0_1200_act-as-security-specialist.js', 'files/javascript/codes/e707046ddb90bb77ad490983a7331415_0_1200_act-as-security-specialist.js', 'files/javascript/codes/e707046ddb90bb77ad490983a7331415_1_1200_act-as-security-specialist.js', 'files/javascript/codes/c081f9f1c3bbbe3b92c3a81ec5c949c5_0_1200_act-as-security-specialist.js', 'files/javascript/codes/c081f9f1c3bbbe3b92c3a81ec5c949c5_1_1200_act-as-security-specialist.js', 'files/javascript/codes/c081f9f1c3bbbe3b92c3a81ec5c949c5_2_1200_act-as-security-specialist.js']
✅ Linting complete, results saved to ..

In [ ]:
# debian based systems
!sudo apt install default-jdk
# or fedora
!sudo dnf install java-21-openjdk-devel.x86_64
# windows:‌ figure it out yourself


[sudo] password for regularpooria: 
^C


In [47]:
import os
import shutil
import hashlib
import re

directory = "../../files/java/codes/"
good_dir = os.path.join(directory, "good")
os.makedirs(good_dir, exist_ok=True)

public_class_pattern = re.compile(r'\bpublic\s+class\s+(\w+)\b')

def hash_file_content(path):
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

mapping = {}

for filename in os.listdir(directory):
    if not filename.endswith(".java"):
        continue

    filepath = os.path.join(directory, filename)
    with open(filepath, encoding="utf-8", errors="ignore") as f:
        content = f.read()

    match = public_class_pattern.search(content)
    if not match:
        continue

    class_name = match.group(1)
    file_hash = hash_file_content(filepath)
    short_hash = file_hash[:8]

    # Create isolated directory per file
    dest_subdir = os.path.join(good_dir, short_hash)
    os.makedirs(dest_subdir, exist_ok=True)

    dest_path = os.path.join(dest_subdir, f"{class_name}.java")

    print(f"Copying {filename} → good/{short_hash}/{class_name}.java")
    shutil.copy2(filepath, dest_path)

    mapping[file_hash] = {
        "original": filename,
        "class": class_name,
        "path": dest_path,
    }


Copying 10a1b82752ca064c385bbb8f17ade3bc_0_1200_act-as-security-specialist.java → good/7b460751/Employee.java
Copying 10a1b82752ca064c385bbb8f17ade3bc_2_1200_act-as-security-specialist.java → good/633ee005/DragDropGUI.java
Copying 10a1b82752ca064c385bbb8f17ade3bc_3_1200_act-as-security-specialist.java → good/b060eaf8/XmlUtil.java
Copying 10a1b82752ca064c385bbb8f17ade3bc_4_1200_act-as-security-specialist.java → good/47502193/PdfWordIndexer.java
Copying 10a1b82752ca064c385bbb8f17ade3bc_5_1200_act-as-security-specialist.java → good/e3289300/H2DatabaseExample.java
Copying 29bce683318c5caec26a230a5fa5466d_0_1200_act-as-security-specialist.java → good/900f72f3/ClassBrowser.java
Copying 29bce683318c5caec26a230a5fa5466d_1_1200_act-as-security-specialist.java → good/b6b34ad6/DeepLearningNetCustomizer.java
Copying 8136fda6374b8b7211623a16dabf0ac4_0_1200_act-as-security-specialist.java → good/d9ea8a2f/MouseRecorder.java
Copying 06ac3ad39fc850bcfabe99e11829f561_0_1200_act-as-security-specialist.ja

In [48]:

# ---- paths ----
directory = "../../files/java/codes/good"
output_file = "../../results/java_lint_conversation_redo.json"
tmp_out = "/tmp/java_linting"

os.makedirs(os.path.dirname(output_file), exist_ok=True)
os.makedirs(tmp_out, exist_ok=True)

# ---- mapping produced by the first script ----
# mapping[file_hash] = {
#     "original": filename,
#     "class": class_name,
#     "path": dest_path,
# }
hash_to_original = mapping

# ---- collect java files recursively ----
java_files = []
for root, _, files in os.walk(directory):
    for f in files:
        if f.endswith(".java"):
            java_files.append(os.path.join(root, f))

all_results = []

# ---- helper: attach errors to original file ----
def add_error(file_path, errors):
    # file_path = .../good/<hash>/ClassName.java
    parts = os.path.normpath(file_path).split(os.sep)

    original = "unknown"
    try:
        hash_dir = parts[-2]
        original = hash_to_original.get(hash_dir, {}).get("original", "unknown")
    except Exception:
        pass

    all_results.append({
        "type": "syntax-error",
        "module": os.path.basename(original),
        "obj": "",
        "line": None,
        "column": None,
        "path": original,
        "symbol": "syntax-error",
        "message": "\n".join(errors),
        "message-id": "",
    })

# ---- run javac ----
result = subprocess.run(
    ["javac", "-d", tmp_out, *java_files],
    capture_output=True,
    text=True,
)

# ---- parse compiler output ----
if result.returncode != 0:
    current_file = None
    current_errors = []

    for line in result.stderr.splitlines():
        if ".java:" in line and "error:" in line:
            if current_file is not None:
                add_error(current_file, current_errors)
            current_errors = [line]
            current_file = line.split(":")[0]
        else:
            current_errors.append(line)

    if current_file is not None:
        add_error(current_file, current_errors)

# ---- write results ----
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)

In [49]:
# Setup directories
directory = "../../files/c/codes/"
output_file = "../../results/c_lint_conversation_redo.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Gather all .c files
c_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".c")]
all_results = []

# Run GCC in syntax-only mode on all files
result = subprocess.run(
    ["gcc", "-fsyntax-only", *c_files],
    capture_output=True,
    text=True,
)

# Parse output if there were errors
if result.returncode != 0:
    for line in result.stderr.splitlines():
        if ": error:" in line:
            parts = line.split(":")
            file_path = parts[0]
            line_no = int(parts[1]) if parts[1].isdigit() else None
            column_no = int(parts[2]) if parts[2].isdigit() else None
            message = line.split(": error:")[1].strip()
            
            # Decode any unicode punctuation to ASCII equivalents
            message = message.replace("\u2018", "'").replace("\u2019", "'")
            
            all_results.append({
                "type": "syntax-error",
                "module": os.path.basename(file_path),
                "obj": "",
                "line": line_no,
                "column": column_no,
                "path": file_path,
                "symbol": "syntax-error",
                "message": message,
                "message-id": "",
            })

# Write results to file
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)

In [51]:

# Setup directories
directory = "../../files/php/codes/"
output_file = "../../results/php_lint_conversation_redo.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Gather all .php files
php_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".php")]
all_results = []

for file in php_files:
    with open(file, "r", encoding="utf-8") as f:
        raw = f.read()
        if "<?php" not in raw:
            all_results.append({
            "type": "syntax-error",
            "module": os.path.basename(file),
            "obj": "",
            "line": None,
            "column": None,
            "path": file,
            "symbol": "Not valid php file",
            "message": "The file does not include <?php tag",
            "message-id": "",
            })
            continue      
            
    result = subprocess.run(
        ["php", "-l", file],
        capture_output=True,
        text=True
    )
    if "Parse error" in result.stdout or "Parse error" in result.stderr:
        match = re.search(r'PHP Parse error: (.+) in .+ on line (\d+)', result.stdout)
        if match:
            message = match.group(1)
            line_no = int(match.group(2))
        else:
            message = result.stdout.strip()
            line_no = 0
        
        all_results.append({
            "type": "syntax-error",
            "module": os.path.basename(file),
            "obj": "",
            "line": line_no,
            "column": None,
            "path": file,
            "symbol": "syntax-error",
            "message": result.stderr + result.stdout,
            "message-id": "",
        })

# Write results to file
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)


In [60]:
import os
import json
import subprocess
from tqdm import tqdm

# Setup directories
directory = "files/csharp/codes/"
output_file = "../../results/csharp_lint_conversation_redo.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Gather all .cs files
csharp_files = [
    os.path.join(directory, f)
    for f in os.listdir("../../"+directory)
    if f.endswith(".cs")
]

all_results = []
batch_size = 500   # tune this as needed (100–1000 is good)

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

# Add tqdm for progress
for batch in tqdm(list(chunked(csharp_files, batch_size)), desc="Checking C# files"):
    result = subprocess.run(
        ["dotnet-script", "utils/SyntaxCheck.csx", *batch],
        capture_output=True,
        text=True,
        cwd="../../",
        env=env
    )

    try:
        batch_results = json.loads(result.stdout)
        all_results.extend(batch_results)
    except json.JSONDecodeError:
        all_results.append({
            "type": "syntax-error",
            "module": "batch",
            "obj": "",
            "line": None,
            "column": None,
            "path": None,
            "symbol": "syntax-error",
            "message": result.stderr + result.stdout,
            "message-id": "",
        })

print(f"Collected {len(all_results)} results")

# Save results
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)


Checking C# files: 100%|██████████| 4/4 [00:02<00:00,  1.68it/s]

Collected 10163 results


## Removing bad lints

In [ ]:
DIRECTORY = "../../results/"
linting_paths = {
    "c" : DIRECTORY + "c_lint_conversation_redo.json",
    "csharp": DIRECTORY + "csharp_lint_conversation_redo.json",
    "java": DIRECTORY + "java_lint_conversation_redo.json",
    "javascript": DIRECTORY + "js_lint_conversation_redo.json",
    "php": DIRECTORY + "php_lint_conversation_redo.json",
    "python": DIRECTORY + "python_lint_conversation_redo.json"
}

In [ ]:
for linting_path in linting_paths:
    with open(linting_path, "r", encoding="utf-8") as f:
        linting_data = json.load(f)

    for row in linting_data:
        filename = row["module"]
        
        # Extract data
        conversation_hash = filename.split("_")[0]
        code_index = filename.split("_")[1].split(".")[0]
        
        for idx, code_snippet in enumerate(code_snippets):
            if code_snippet["conversation_hash"] == conversation_hash and code_snippet["code_index"] == code_index:
                del code_snippets[idx]
                break